# E3: Structure-Aware Transformer for DNA Thermodynamics

**Thesis:** Inductive Biases in Representation Learning for DNA Thermodynamic Property Prediction  
**Experiment ID:** E3  
**Thesis Chapter:** Chapter 4 — Attention vs. Message Passing  

## Core Idea

Standard Transformers treat all positions as equally reachable via self-attention, but for a DNA hairpin, bases that are **hydrogen-bonded** (e.g., position 2 and position 18) are physically closer than backbone-adjacent bases. We inject this structural knowledge directly into the attention computation:

$$\text{Attention}(Q,K,V) = \text{Softmax}\!\left(\frac{QK^\top}{\sqrt{d}} + \lambda \cdot M\right) V$$

where **M** is a binary adjacency matrix (1 = H-bond or backbone neighbor, 0 otherwise) derived from the dot-bracket secondary structure, and **λ** is a **learnable scalar** that controls how strongly the model trusts the structural prior.

### Inductive Bias Being Tested
- **GNN (E0):** permutation equivariance via local message passing  
- **2D CNN (E2):** spatial translation invariance on folded grid  
- **SAT (E3, this notebook):** explicit relational attention — the model is told *which pairs matter*, but learns *how much* they matter via λ  

### Research Question
> Does injecting a structural relational bias into the Transformer attention mechanism outperform both GNN message-passing and 2D spatial convolution for DNA thermodynamic regression?


In [1]:
# ── 1. Setup & Imports ────────────────────────────────────────────────────────
import os, json, math, time
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

import wandb

# ── WandB: match the pattern used in 1D_CNN_for_dna.ipynb ────────────────────
# On Windows Jupyter, force online mode so credentials from ~/.netrc are used.
# To run fully offline: set env var WANDB_MODE=disabled before starting Jupyter.
import sys
if sys.platform == 'win32' and not os.environ.get('WANDB_MODE'):
    os.environ['WANDB_MODE'] = 'online'

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Thesis colour palette (expdesign.md §4) ───────────────────────────────────
COLORS = {
    'GNN':    '#7f8c8d',
    '1D_CNN': '#3498db',
    '2D_CNN': '#e74c3c',
    'SAT':    '#9b59b6',   # purple for Structure-Aware Transformer
    'PINN':   '#e67e22',
}
MODEL_NAME = 'SAT'

c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda:0
GPU: NVIDIA GeForce GTX 1660 Ti


In [2]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
MAX_LEN = 24
BASES   = {'A': 0, 'T': 1, 'G': 2, 'C': 3}
STRUCTS = {'(': 0, ')': 1, '.': 2}

DATA_CSV   = 'data/models/raw/combined_dataset.csv'
SPLIT_JSON = 'data/models/raw/combined_data_split.json'

config = dict(
    model_name       = 'StructureAwareTransformer',
    experiment_id    = 'E3',
    max_len          = MAX_LEN,
    input_dim        = 7,          # 4 one-hot base + 3 one-hot structure
    d_model          = 128,
    nhead            = 8,
    num_layers       = 4,
    ff_dim           = 256,
    dropout          = 0.1,
    lambda_init      = 1.0,        # initial structural bias weight
    n_epoch          = 200,
    batch_size       = 256,
    lr               = 1e-3,
    weight_decay     = 1e-5,
    grad_clip        = 1.0,
    dataset          = 'arr',
    norm_method      = 'normalize',
    wandb_project    = 'NNN_Thesis_Experiments',
    checkpoint_dir   = 'MyExperiments/SAT/models',
)
print('Config loaded:', config)

Config loaded: {'model_name': 'StructureAwareTransformer', 'experiment_id': 'E3', 'max_len': 24, 'input_dim': 7, 'd_model': 128, 'nhead': 8, 'num_layers': 4, 'ff_dim': 256, 'dropout': 0.1, 'lambda_init': 1.0, 'n_epoch': 200, 'batch_size': 256, 'lr': 0.001, 'weight_decay': 1e-05, 'grad_clip': 1.0, 'dataset': 'arr', 'norm_method': 'normalize', 'wandb_project': 'NNN_Thesis_Experiments', 'checkpoint_dir': 'MyExperiments/SAT/models'}


In [3]:
# ── 3. Data Loading & Normalization ───────────────────────────────────────────
df = pd.read_csv(DATA_CSV, index_col='SEQID')
df.sort_index(inplace=True)

with open(SPLIT_JSON) as f:
    split = json.load(f)

# Train & evaluate on 'arr' only — lit_uv / ov are held-out generalization sets
TRAIN_DATASET = 'arr'

train_df = df.loc[split['train_ind']].dropna(subset=['dH', 'Tm'])
train_df = train_df[train_df['dataset'] == TRAIN_DATASET]
val_df   = df.loc[split['val_ind']  ].dropna(subset=['dH', 'Tm'])
val_df   = val_df[val_df['dataset'] == TRAIN_DATASET]
test_df  = df.loc[split['test_ind'] ].dropna(subset=['dH', 'Tm'])
test_df  = test_df[test_df['dataset'] == TRAIN_DATASET]

# Compute normalisation stats from TRAINING set only
sumstats = {
    'dH_min': train_df['dH'].min(), 'dH_max': train_df['dH'].max(),
    'Tm_min': train_df['Tm'].min(), 'Tm_max': train_df['Tm'].max(),
}

def normalize(val, mn, mx):   return (val - mn) / (mx - mn)
def unnormalize(val, mn, mx): return val * (mx - mn) + mn

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}  (arr only)')
print(f'dH  range: [{sumstats["dH_min"]:.1f}, {sumstats["dH_max"]:.1f}] kcal/mol')
print(f'Tm  range: [{sumstats["Tm_min"]:.1f}, {sumstats["Tm_max"]:.1f}] °C')

Train: 25,025  |  Val: 1,318  |  Test: 1,387  (arr only)
dH  range: [-68.2, -2.7] kcal/mol
Tm  range: [13.6, 68.6] °C


In [4]:
# ── 4. Encoding: Sequence + Structure → Tensor + Structural Bias Matrix ───────

def encode_seq_struct(seq, struct, max_len=MAX_LEN):
    """
    Returns (max_len, 7) float32 array.
    Channels 0-3: one-hot nucleotide (A,T,G,C)
    Channels 4-6: one-hot structure symbol ( ( ) . )
    Padding positions (beyond seq length) are all zeros.
    """
    L = min(len(seq), max_len)
    x = np.zeros((max_len, 7), dtype=np.float32)
    for i in range(L):
        if seq[i] in BASES:
            x[i, BASES[seq[i]]] = 1.0
        if struct[i] in STRUCTS:
            x[i, 4 + STRUCTS[struct[i]]] = 1.0
    return x


def build_struct_bias(struct, max_len=MAX_LEN):
    """
    Build structural adjacency matrix M of shape (max_len, max_len).

    M[i,j] = 1  if i and j are hydrogen-bonded (paired in dot-bracket)
    M[i,j] = 1  if |i-j| == 1  (backbone neighbours)
    M[i,j] = 0  otherwise (including padding)

    This matrix is added as a bias to attention logits:
        logits += lambda * M
    so paired and adjacent positions are encouraged to attend to each other.
    """
    L = min(len(struct), max_len)
    M = np.zeros((max_len, max_len), dtype=np.float32)

    # Backbone connectivity
    for i in range(L - 1):
        M[i, i + 1] = 1.0
        M[i + 1, i] = 1.0

    # H-bond connectivity from dot-bracket
    stack = []
    for i, c in enumerate(struct[:L]):
        if c == '(':
            stack.append(i)
        elif c == ')' and stack:
            j = stack.pop()
            M[i, j] = 1.0
            M[j, i] = 1.0

    return M


# Quick sanity check
_seq    = 'GCGCTTTTGCGC'
_struct = '((((....))))'
_x = encode_seq_struct(_seq, _struct)
_M = build_struct_bias(_struct)
print(f'Encoding shape: {_x.shape}  |  Bias matrix shape: {_M.shape}')
print(f'Paired positions in M: {(_M[:len(_seq), :len(_seq)] > 0).sum()} (backbone + H-bonds)')

Encoding shape: (24, 7)  |  Bias matrix shape: (24, 24)
Paired positions in M: 30 (backbone + H-bonds)


In [ ]:
# ── 5. Dataset & DataLoaders ───────────────────────────────────

class DNASATDataset(Dataset):
    """PyTorch Dataset for Structure-Aware Transformer."""

    def __init__(self, df, sumstats, max_len=MAX_LEN):
        self.df       = df.reset_index(drop=False)
        self.sumstats = sumstats
        self.max_len  = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        seq    = str(row['RefSeq'])
        struct = str(row['TargetStruct'])

        x = encode_seq_struct(seq, struct, self.max_len)   # (L, 7)
        M = build_struct_bias(struct, self.max_len)        # (L, L)

        dH_norm = normalize(row['dH'], self.sumstats['dH_min'], self.sumstats['dH_max'])
        Tm_norm = normalize(row['Tm'], self.sumstats['Tm_min'], self.sumstats['Tm_max'])
        y = np.array([dH_norm, Tm_norm], dtype=np.float32)

        return torch.tensor(x), torch.tensor(M), torch.tensor(y)


train_ds = DNASATDataset(train_df, sumstats)
val_ds   = DNASATDataset(val_df,sumstats)
test_ds  = DNASATDataset(test_df,  sumstats)

train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=512,                  shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=512,                  shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')
_x, _M, _y = next(iter(train_loader))
print(f'Batch shapes — x: {_x.shape}  M: {_M.shape}  y: {_y.shape}')

Train batches: 98  |  Val batches: 3
Batch shapes — x: torch.Size([256, 24, 7])  M: torch.Size([256, 24, 24])  y: torch.Size([256, 2])


In [6]:
# ── 6. Model: Structure-Aware Transformer ─────────────────────────────────────

class StructureBiasedAttention(nn.Module):
    """
    Multi-head self-attention with an additive structural bias.

    Standard scaled dot-product attention:
        Attn(Q,K,V) = Softmax(QK^T / sqrt(d_head)) V

    Structure-biased version:
        Attn(Q,K,V) = Softmax(QK^T / sqrt(d_head) + lambda * M) V

    lambda is a learnable scalar per layer (shared across heads).
    M is the structural adjacency matrix (backbone + H-bonds).
    """

    def __init__(self, d_model, nhead, dropout=0.1, lambda_init=1.0):
        super().__init__()
        assert d_model % nhead == 0, 'd_model must be divisible by nhead'
        self.d_model  = d_model
        self.nhead    = nhead
        self.head_dim = d_model // nhead
        self.scale    = math.sqrt(self.head_dim)

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)

        # Learnable structural bias weight
        self.lambda_bias = nn.Parameter(torch.tensor(float(lambda_init)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, M, pad_mask=None):
        """
        x:        (B, L, d_model)
        M:        (B, L, L)   structural adjacency (float)
        pad_mask: (B, L)      True for padded (zero) positions
        Returns:  (B, L, d_model)
        """
        B, L, _ = x.shape
        H, D = self.nhead, self.head_dim

        # Project and split into heads
        Q = self.W_q(x).view(B, L, H, D).transpose(1, 2)  # (B,H,L,D)
        K = self.W_k(x).view(B, L, H, D).transpose(1, 2)
        V = self.W_v(x).view(B, L, H, D).transpose(1, 2)

        # Scaled dot-product logits
        logits = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # (B,H,L,L)

        # Add structural bias: lambda * M  (broadcast over heads)
        logits = logits + self.lambda_bias * M.unsqueeze(1)  # (B,H,L,L)

        # Mask padding positions with -inf
        if pad_mask is not None:
            logits = logits.masked_fill(
                pad_mask.unsqueeze(1).unsqueeze(2), float('-inf')
            )

        attn = torch.softmax(logits, dim=-1)     # (B,H,L,L)
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)              # (B,H,L,D)
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.W_o(out), attn


class SATransformerLayer(nn.Module):
    """Pre-LayerNorm Transformer encoder layer with structure-biased attention."""

    def __init__(self, d_model, nhead, ff_dim, dropout=0.1, lambda_init=1.0):
        super().__init__()
        self.attn  = StructureBiasedAttention(d_model, nhead, dropout, lambda_init)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, M, pad_mask=None):
        # Self-attention with residual (pre-norm)
        attn_out, attn_w = self.attn(self.norm1(x), M, pad_mask)
        x = x + self.drop(attn_out)
        # Feed-forward with residual
        x = x + self.drop(self.ff(self.norm2(x)))
        return x, attn_w


class StructureAwareTransformer(nn.Module):
    """
    Full Structure-Aware Transformer for DNA thermodynamic regression.

    Architecture:
        Input projection  (L, 7) → (L, d_model)
        Learned positional encoding
        N x SATransformerLayer  (structure-biased self-attention)
        Masked mean pooling     → (d_model,)
        MLP head                → [dH_norm, Tm_norm]
    """

    def __init__(self, input_dim=7, d_model=128, nhead=8, num_layers=4,
                 ff_dim=256, dropout=0.1, max_len=MAX_LEN, lambda_init=1.0):
        super().__init__()
        self.d_model = d_model

        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_embed  = nn.Embedding(max_len, d_model)
        self.pos_drop   = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            SATransformerLayer(d_model, nhead, ff_dim, dropout, lambda_init)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, 2),
        )
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x, M):
        """
        x: (B, L, 7)    sequence + structure one-hot features
        M: (B, L, L)    structural adjacency matrix
        Returns: (B, 2)  [dH_norm, Tm_norm]
        """
        B, L, _ = x.shape
        pos     = torch.arange(L, device=x.device).unsqueeze(0).expand(B, -1)
        pad_mask = (x.sum(-1) == 0)   # (B, L) True for all-zero padding

        h = self.pos_drop(self.input_proj(x) + self.pos_embed(pos))  # (B,L,d)

        self._last_attn_weights = []
        for layer in self.layers:
            h, attn_w = layer(h, M, pad_mask)
            self._last_attn_weights.append(attn_w.detach())

        h = self.norm(h)

        # Masked mean pooling (ignore padding)
        not_pad = (~pad_mask).float().unsqueeze(-1)     # (B,L,1)
        h = (h * not_pad).sum(1) / not_pad.sum(1).clamp(min=1)  # (B,d)

        return self.head(h)   # (B,2)


# ── Instantiate ───────────────────────────────────────────────────────────────
model = StructureAwareTransformer(
    input_dim  = config['input_dim'],
    d_model    = config['d_model'],
    nhead      = config['nhead'],
    num_layers = config['num_layers'],
    ff_dim     = config['ff_dim'],
    dropout    = config['dropout'],
    max_len    = config['max_len'],
    lambda_init= config['lambda_init'],
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: StructureAwareTransformer')
print(f'Total trainable parameters: {n_params:,}')
print(f'  d_model={config["d_model"]}, nhead={config["nhead"]}, layers={config["num_layers"]}, ff_dim={config["ff_dim"]}')
print(f'Initial lambda_bias values: { [f"{l.attn.lambda_bias.item():.3f}" for l in model.layers] }')

Model: StructureAwareTransformer
Total trainable parameters: 541,126
  d_model=128, nhead=8, layers=4, ff_dim=256
Initial lambda_bias values: ['1.000', '1.000', '1.000', '1.000']


In [7]:
# ── 7. Metrics & Evaluation Helpers (standard across all notebooks) ───────────

def compute_metrics(pred_norm, true_norm, sumstats):
    """
    Compute MAE, RMSE, R² for dH, Tm, dG_37 in original physical units.
    pred_norm, true_norm: (N, 2) numpy arrays [dH_norm, Tm_norm]
    Returns: metrics dict + unnormalized arrays
    """
    if torch.is_tensor(pred_norm):
        pred_norm = pred_norm.cpu().numpy()
    if torch.is_tensor(true_norm):
        true_norm = true_norm.cpu().numpy()

    dH_p = unnormalize(pred_norm[:, 0], sumstats['dH_min'], sumstats['dH_max'])
    Tm_p = unnormalize(pred_norm[:, 1], sumstats['Tm_min'], sumstats['Tm_max'])
    dH_t = unnormalize(true_norm[:, 0], sumstats['dH_min'], sumstats['dH_max'])
    Tm_t = unnormalize(true_norm[:, 1], sumstats['Tm_min'], sumstats['Tm_max'])

    # dG_37 derived from dH and Tm
    dG_p = dH_p * (1.0 - (273.15 + 37.0) / (273.15 + Tm_p))
    dG_t = dH_t * (1.0 - (273.15 + 37.0) / (273.15 + Tm_t))

    metrics = {}
    for tag, p, t in [('dH', dH_p, dH_t), ('Tm', Tm_p, Tm_t), ('dG_37', dG_p, dG_t)]:
        # NaN-safe: some external datasets have missing dH measurements
        mask = np.isfinite(t) & np.isfinite(p)
        if mask.sum() < 2:
            metrics[f'{tag}_mae']  = float('nan')
            metrics[f'{tag}_rmse'] = float('nan')
            metrics[f'{tag}_r2']   = float('nan')
        else:
            diff = p[mask] - t[mask]
            metrics[f'{tag}_mae']  = float(np.mean(np.abs(diff)))
            metrics[f'{tag}_rmse'] = float(np.sqrt(np.mean(diff ** 2)))
            metrics[f'{tag}_r2']   = float(r2_score(t[mask], p[mask]))

    return metrics, dH_p, Tm_p, dH_t, Tm_t


@torch.no_grad()
def evaluate(model, loader, sumstats, device):
    """Run model over loader, return metrics + raw arrays."""
    model.eval()
    preds, trues = [], []
    for x, M, y in loader:
        x, M = x.to(device), M.to(device)
        preds.append(model(x, M).cpu())
        trues.append(y)
    preds = torch.cat(preds)
    trues = torch.cat(trues)
    metrics, dH_p, Tm_p, dH_t, Tm_t = compute_metrics(preds, trues, sumstats)
    return metrics, dH_p, Tm_p, dH_t, Tm_t


print('Metrics helpers defined.')

Metrics helpers defined.


In [8]:
# ── 8. Training Loop ──────────────────────────────────────────────────────────

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=config['lr'],
                       weight_decay=config['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config['n_epoch'], eta_min=1e-5
)

# History dict — identical schema across all notebooks for combined F2 figure
history = {
    'train_loss': [],
    'val_dH_mae': [], 'val_Tm_mae': [], 'val_dG_mae': [],
    'val_dH_rmse': [], 'val_Tm_rmse': [],
}

os.makedirs(config['checkpoint_dir'], exist_ok=True)
os.makedirs('out', exist_ok=True)

# ── WandB init: try online (credentials from ~/.netrc), fall back to offline ──
_run_name   = f"SAT_d{config['d_model']}_h{config['nhead']}_L{config['num_layers']}"
_wandb_kw   = dict(project=config['wandb_project'], name=_run_name, config=config, reinit=True)
_wandb_mode = os.environ.get('WANDB_MODE', '').strip().lower()
if _wandb_mode in ('offline', 'disabled'):
    run = wandb.init(mode=_wandb_mode, **_wandb_kw)
else:
    try:
        run = wandb.init(**_wandb_kw)
    except Exception as _e:
        print(f'WandB online init failed ({_e}), falling back to offline mode.')
        run = wandb.init(mode='offline', **_wandb_kw)
print(f'WandB run: {run.name}  |  mode: {run.settings.mode}')

best_val_dG = float('inf')
start_time  = time.time()

for epoch in range(config['n_epoch']):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for x, M, y in train_loader:
        x, M, y = x.to(device), M.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x, M), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_loader.dataset)
    scheduler.step()

    # ── Validate ──────────────────────────────────────────────────────────────
    val_metrics, _, _, _, _ = evaluate(model, val_loader, sumstats, device)

    history['train_loss'].append(train_loss)
    history['val_dH_mae'].append(val_metrics['dH_mae'])
    history['val_Tm_mae'].append(val_metrics['Tm_mae'])
    history['val_dG_mae'].append(val_metrics['dG_37_mae'])
    history['val_dH_rmse'].append(val_metrics['dH_rmse'])
    history['val_Tm_rmse'].append(val_metrics['Tm_rmse'])

    # Track learnable lambda across layers
    lambdas = {f'lambda_L{i}': l.attn.lambda_bias.item()
               for i, l in enumerate(model.layers)}

    wandb.log({
        'epoch': epoch,
        'train_loss': train_loss,
        **{f'val_{k}': v for k, v in val_metrics.items()},
        'lr': scheduler.get_last_lr()[0],
        **lambdas,
    })

    # ── Checkpoint best model ─────────────────────────────────────────────────
    if val_metrics['dG_37_mae'] < best_val_dG:
        best_val_dG = val_metrics['dG_37_mae']
        torch.save(model.state_dict(),
                   os.path.join(config['checkpoint_dir'], 'best_sat_model.pt'))

    if (epoch + 1) % 20 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"Ep {epoch+1:3d}/{config['n_epoch']} "
              f"| loss {train_loss:.4f} "
              f"| dH {val_metrics['dH_mae']:.3f} "
              f"| Tm {val_metrics['Tm_mae']:.3f} "
              f"| dG {val_metrics['dG_37_mae']:.3f} "
              f"| λ₀={model.layers[0].attn.lambda_bias.item():.3f} "
              f"| {elapsed:.1f}min")

run.finish()

# Save history for combined F2 figure generation
import json as _json
with open('out/sat_history.json', 'w') as f:
    _json.dump(history, f)

print(f'\n=== Training complete ===')
print(f'Best val dG_37 MAE: {best_val_dG:.4f} kcal/mol')
print(f'History saved to: out/sat_history.json')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\anant\.netrc.
wandb: Currently logged in as: apati087 (apati087-university-of-california-riverside) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


WandB run: SAT_d128_h8_L4  |  mode: online
Ep  20/200 | loss 0.0076 | dH 4.049 | Tm 3.066 | dG 0.276 | λ₀=1.129 | 3.3min
Ep  40/200 | loss 0.0059 | dH 3.473 | Tm 2.753 | dG 0.246 | λ₀=1.101 | 6.7min
Ep  60/200 | loss 0.0052 | dH 3.727 | Tm 2.587 | dG 0.250 | λ₀=1.067 | 10.1min
Ep  80/200 | loss 0.0049 | dH 3.594 | Tm 2.457 | dG 0.237 | λ₀=1.038 | 13.4min
Ep 100/200 | loss 0.0045 | dH 3.540 | Tm 2.478 | dG 0.237 | λ₀=1.018 | 17.3min
Ep 120/200 | loss 0.0043 | dH 3.597 | Tm 2.430 | dG 0.254 | λ₀=1.002 | 21.0min
Ep 140/200 | loss 0.0041 | dH 3.415 | Tm 2.353 | dG 0.232 | λ₀=0.991 | 24.4min
Ep 160/200 | loss 0.0040 | dH 3.435 | Tm 2.353 | dG 0.229 | λ₀=0.980 | 27.5min
Ep 180/200 | loss 0.0039 | dH 3.384 | Tm 2.271 | dG 0.230 | λ₀=0.972 | 30.8min
Ep 200/200 | loss 0.0039 | dH 3.387 | Tm 2.271 | dG 0.229 | λ₀=0.970 | 34.2min


epoch,▁▁▁▁▁▁▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
lambda_L0,▄▆▇██▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
lambda_L1,██▇▇▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lambda_L2,█▇▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lambda_L3,█▅▄▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lr,█████▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁
train_loss,█▆▅▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_Tm_mae,▇██▄▄▅▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▂▂▂▁▁▁▁▁▁▁▁▁▁
val_Tm_r2,▁▃▄▂▄▅▅▆▇▇▇▇▇▇▇▇▇▆▇▇██▇█████████████████
val_Tm_rmse,██▅▄▄▃▃▃▄▃▂▂▂▂▂▂▁▂▁▂▁▂▂▂▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁
+6,...



=== Training complete ===
Best val dG_37 MAE: 0.2173 kcal/mol
History saved to: out/sat_history.json


In [9]:
# ── 9. Final Evaluation on Val + Test ─────────────────────────────────────────

# Load best checkpoint
model.load_state_dict(torch.load(
    os.path.join(config['checkpoint_dir'], 'best_sat_model.pt'),
    map_location=device
))

val_metrics,  dH_vp, Tm_vp, dH_vt, Tm_vt = evaluate(model, val_loader,  sumstats, device)
test_metrics, dH_tp, Tm_tp, dH_tt, Tm_tt = evaluate(model, test_loader, sumstats, device)

print('=== Validation Set Results (arr) ===')
print(f'  dH   MAE  {val_metrics["dH_mae"]:.3f}  RMSE {val_metrics["dH_rmse"]:.3f}  R² {val_metrics["dH_r2"]:.3f}')
print(f'  Tm   MAE  {val_metrics["Tm_mae"]:.3f}  RMSE {val_metrics["Tm_rmse"]:.3f}  R² {val_metrics["Tm_r2"]:.3f}')
print(f'  dG37 MAE  {val_metrics["dG_37_mae"]:.3f}  RMSE {val_metrics["dG_37_rmse"]:.3f}  R² {val_metrics["dG_37_r2"]:.3f}')

print('\n=== Test Set Results (arr) ===')
print(f'  dH   MAE  {test_metrics["dH_mae"]:.3f}  RMSE {test_metrics["dH_rmse"]:.3f}  R² {test_metrics["dH_r2"]:.3f}')
print(f'  Tm   MAE  {test_metrics["Tm_mae"]:.3f}  RMSE {test_metrics["Tm_rmse"]:.3f}  R² {test_metrics["Tm_r2"]:.3f}')
print(f'  dG37 MAE  {test_metrics["dG_37_mae"]:.3f}  RMSE {test_metrics["dG_37_rmse"]:.3f}  R² {test_metrics["dG_37_r2"]:.3f}')

# Save eval CSV for combined F4 figure
eval_df = pd.DataFrame({
    'dH_pred': dH_vp, 'dH_true': dH_vt,
    'Tm_pred': Tm_vp, 'Tm_true': Tm_vt,
})
eval_df['dG_pred'] = eval_df['dH_pred'] * (1 - 310.15 / (273.15 + eval_df['Tm_pred']))
eval_df['dG_true'] = eval_df['dH_true'] * (1 - 310.15 / (273.15 + eval_df['Tm_true']))
eval_df.to_csv('out/sat_val_eval.csv', index=False)
print('\nVal predictions saved to: out/sat_val_eval.csv')

=== Validation Set Results (arr) ===
  dH   MAE  3.419  RMSE 4.601  R² 0.826
  Tm   MAE  2.251  RMSE 3.277  R² 0.908
  dG37 MAE  0.217  RMSE 0.314  R² 0.908

=== Test Set Results (arr) ===
  dH   MAE  3.264  RMSE 4.299  R² 0.848
  Tm   MAE  2.320  RMSE 3.303  R² 0.903
  dG37 MAE  0.214  RMSE 0.295  R² 0.920

Val predictions saved to: out/sat_val_eval.csv


In [10]:
# ── 10. Convergence Curves (F2 contribution) ──────────────────────────────────
# This cell generates the SAT-only convergence plot.
# Combined F2 (all models) is generated by gen_F2_F3_F4.py once all notebooks are run.

os.makedirs('out/figures', exist_ok=True)
epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), facecolor='#f8f9fa')
sns.set_palette([COLORS['SAT']])

metrics_to_plot = [
    ('val_dH_mae',  'Validation dH MAE (kcal/mol)',  'ΔH'),
    ('val_Tm_mae',  'Validation Tm MAE (°C)',         'Tm'),
    ('val_dG_mae',  'Validation ΔG₃₇ MAE (kcal/mol)', 'ΔG₃₇'),
]

for ax, (key, ylabel, title) in zip(axes, metrics_to_plot):
    ax.plot(epochs, history[key], color=COLORS['SAT'], lw=2, label='SAT (E3)')
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    sns.despine(ax=ax)

fig.suptitle('Figure F2 (partial) — SAT Validation Convergence', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('out/figures/sat_convergence.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: out/figures/sat_convergence.png')

c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 8323 missing from current font.
  font.set_text(s, 0.0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 8327 missing from current font.
  font.set_text(s, 0.0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:203: RuntimeWarning: Glyph 8323 missing from current font.
  font.set_text(s, 0, flags=flags)
c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\matplotlib\backends\backend_agg.py:203: RuntimeWarning: Glyph 8327 missing from current font.
  font.set_text(s, 0, flags=flags)


Saved: out/figures/sat_convergence.png


C:\Users\anant\AppData\Local\Temp\ipykernel_29272\3679999823.py:28: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [11]:
# ── 11. Scatter Plots — Predicted vs Measured (F3 contribution) ───────────────
# Axis limits from expdesign.md §4
AXIS_LIMITS = {'dH': (-55, -5), 'Tm': (20, 60), 'dG_37': (-7, 5)}

fig, axes = plt.subplots(1, 3, figsize=(14, 5), facecolor='#f8f9fa')

dG_vp = dH_vp * (1 - 310.15 / (273.15 + Tm_vp))
dG_vt = dH_vt * (1 - 310.15 / (273.15 + Tm_vt))

for ax, (pred_arr, true_arr, tag, unit) in zip(axes, [
    (dH_vp, dH_vt, 'dH',    'kcal/mol'),
    (Tm_vp, Tm_vt, 'Tm',    '°C'),
    (dG_vp, dG_vt, 'dG_37', 'kcal/mol'),
]):
    lim = AXIS_LIMITS[tag]
    ax.scatter(true_arr, pred_arr, s=4, alpha=0.4,
               color=COLORS['SAT'], rasterized=True)
    ax.plot(lim, lim, 'k--', alpha=0.3, lw=1.5, label='y = x')
    mae  = np.mean(np.abs(pred_arr - true_arr))
    rmse = np.sqrt(np.mean((pred_arr - true_arr) ** 2))
    r2   = r2_score(true_arr, pred_arr)
    ax.text(0.05, 0.93, f'MAE={mae:.3f}\nRMSE={rmse:.3f}\nR²={r2:.3f}',
            transform=ax.transAxes, fontsize=8.5, va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel(f'Measured {tag} ({unit})', fontsize=10)
    ax.set_ylabel(f'Predicted {tag} ({unit})', fontsize=10)
    ax.set_title(f'SAT — {tag}', fontsize=11, fontweight='bold')
    sns.despine(ax=ax)

fig.suptitle('Figure F3 (partial) — SAT: Predicted vs Measured (Validation)', fontsize=12)
plt.tight_layout()
plt.savefig('out/figures/sat_scatter.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: out/figures/sat_scatter.png')

Saved: out/figures/sat_scatter.png


C:\Users\anant\AppData\Local\Temp\ipykernel_29272\549724167.py:34: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [12]:
# ── 12. Attention Visualisation (F5) — What Does the SAT Attend To? ───────────
# Show attention maps for one example: does the model attend to paired bases?

example_idx = 0
x_ex, M_ex, y_ex = val_ds[example_idx]
seq_ex    = val_df.iloc[example_idx]['RefSeq']
struct_ex = val_df.iloc[example_idx]['TargetStruct']
L_ex      = len(seq_ex)

model.eval()
with torch.no_grad():
    _ = model(x_ex.unsqueeze(0).to(device), M_ex.unsqueeze(0).to(device))

# Show last layer attention (averaged over heads)
last_attn = model._last_attn_weights[-1][0]  # (H, L, L)
avg_attn  = last_attn.mean(0).cpu().numpy()[:L_ex, :L_ex]  # (L_ex, L_ex)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='#f8f9fa')

# Attention map
im = axes[0].imshow(avg_attn, cmap='viridis', aspect='auto')
axes[0].set_xticks(range(L_ex)); axes[0].set_xticklabels(list(seq_ex), fontsize=8)
axes[0].set_yticks(range(L_ex)); axes[0].set_yticklabels(list(seq_ex), fontsize=8)
axes[0].set_title(f'SAT Average Attention — Last Layer\n{seq_ex} | {struct_ex}', fontsize=10)
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Structural bias matrix
M_vis = M_ex.numpy()[:L_ex, :L_ex]
im2 = axes[1].imshow(M_vis, cmap='Purples', aspect='auto')
axes[1].set_xticks(range(L_ex)); axes[1].set_xticklabels(list(seq_ex), fontsize=8)
axes[1].set_yticks(range(L_ex)); axes[1].set_yticklabels(list(seq_ex), fontsize=8)
axes[1].set_title(f'Structural Bias Matrix M\n(backbone + H-bonds)', fontsize=10)
plt.colorbar(im2, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.savefig('out/figures/sat_attention_map.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: out/figures/sat_attention_map.png')
print('\nIf the left and right heatmaps show similar high-attention patterns,')
print('the SAT has learned to attend to structurally relevant positions — validating E3.')

Saved: out/figures/sat_attention_map.png

If the left and right heatmaps show similar high-attention patterns,
the SAT has learned to attend to structurally relevant positions — validating E3.


C:\Users\anant\AppData\Local\Temp\ipykernel_29272\2440207462.py:37: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [13]:
# ── 13. Lambda Evolution — Did the Model Learn the Structural Bias? ────────────
# Print final learnable lambda values across all layers
print('=== Learned λ (structural bias weight) per layer ===')
print('(λ > 0: H-bond and backbone pairs attend to each other MORE than random)')
print('(λ < 0: the model IGNORES or suppresses structural pairs)')
print()
for i, layer in enumerate(model.layers):
    lam = layer.attn.lambda_bias.item()
    direction = 'ENHANCES' if lam > 0 else 'SUPPRESSES'
    print(f'  Layer {i}: λ = {lam:+.4f}  →  {direction} structural attention')

=== Learned λ (structural bias weight) per layer ===
(λ > 0: H-bond and backbone pairs attend to each other MORE than random)
(λ < 0: the model IGNORES or suppresses structural pairs)

  Layer 0: λ = +0.9786  →  ENHANCES structural attention
  Layer 1: λ = +0.2403  →  ENHANCES structural attention
  Layer 2: λ = +0.0028  →  ENHANCES structural attention
  Layer 3: λ = +0.0003  →  ENHANCES structural attention


In [14]:
# ── 14. Save Checkpoint Info for EXPERIMENT_LOG ───────────────────────────────
import json as _json

log_entry = {
    'experiment_id': 'E3',
    'model': 'StructureAwareTransformer',
    'config': config,
    'n_params': n_params,
    'val_metrics': val_metrics,
    'test_metrics': test_metrics,
    'best_checkpoint': os.path.join(config['checkpoint_dir'], 'best_sat_model.pt'),
    'history_path': 'out/sat_history.json',
    'eval_csv_path': 'out/sat_val_eval.csv',
    'lambda_final': {f'L{i}': l.attn.lambda_bias.item() for i, l in enumerate(model.layers)},
}

with open('out/sat_run_log.json', 'w') as f:
    _json.dump(log_entry, f, indent=2)

print('Run log saved to: out/sat_run_log.json')
print()
print('=== E3 COMPLETE ===')
print(f'  Val  dH/Tm/dG MAE: {val_metrics["dH_mae"]:.3f} / {val_metrics["Tm_mae"]:.3f} / {val_metrics["dG_37_mae"]:.3f}')
print(f'  Test dH/Tm/dG MAE: {test_metrics["dH_mae"]:.3f} / {test_metrics["Tm_mae"]:.3f} / {test_metrics["dG_37_mae"]:.3f}')
print()
print('Next steps:')
print('  1. Update EXPERIMENT_LOG.md with E3 results')
print('  2. Update Ph3.md: set E3 status = done')
print('  3. Proceed to notebook standardization (Track B) or E4 (PINN)')

Run log saved to: out/sat_run_log.json

=== E3 COMPLETE ===
  Val  dH/Tm/dG MAE: 3.419 / 2.251 / 0.217
  Test dH/Tm/dG MAE: 3.264 / 2.320 / 0.214

Next steps:
  1. Update EXPERIMENT_LOG.md with E3 results
  2. Update Ph3.md: set E3 status = done
  3. Proceed to notebook standardization (Track B) or E4 (PINN)
